# 🍛 Model 1: Food Detection Version 1 Training (YOLOv8)
**South Indian Food Nutrition & Detection AI Project**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dharmenthiran-spi/SouthIndian_Food_Nutrition/blob/main/Food_Detection/NoteBooks/Food%20Detection/Version1/food_detection_v1_training.ipynb)

---
### 🔄 Clean Single-Source Workflow (No Google Drive Sync Issues)
1. **Click 'Open in Colab'** badge above to launch this notebook in Google Colab.
2. **Run All Cells**: Colab freshly clones the repository and trains on ultra-fast local disk (`/content/`).
3. **1-Click Download**: Automatically downloads `food_detection_v1_best.pt` directly to your local computer's **Downloads** folder.
4. **Local Integration**: Place `food_detection_v1_best.pt` in `Food_Detection/Models/Food Detection/Version1/` and launch `streamlit run app.py`!

## Step 1: Environment & Dependency Setup
We install `ultralytics` and verify that a GPU is available for deep learning training.

In [ ]:
# Install Ultralytics and dependencies
!pip install -q ultralytics torchvision opencv-python-headless pyyaml matplotlib seaborn

import torch, sys
import ultralytics

print("=" * 60)
print("🖥️  SYSTEM & HARDWARE SPECIFICATIONS")
print("=" * 60)
print(f"Python Version   : {sys.version.split()[0]}")
print(f"PyTorch Version  : {torch.__version__}")
print(f"Ultralytics      : {ultralytics.__version__}")
print(f"CUDA Available   : {'✅ Yes' if torch.cuda.is_available() else '❌ No (Switch Colab runtime to GPU!)'}")
if torch.cuda.is_available():
    print(f"Device Name      : {torch.cuda.get_device_name(0)}")
print("=" * 60)

ultralytics.checks()

## Step 2: Fresh Repository Clone in Colab (Fast Local Disk)
Clones the latest code directly from GitHub into Colab's fast local SSD storage.

In [ ]:
import os, yaml

REPO_URL = "https://github.com/Dharmenthiran-spi/SouthIndian_Food_Nutrition.git"
REPO_NAME = "SouthIndian_Food_Nutrition"

# In Google Colab, fresh clone
if "google.colab" in str(get_ipython()):
    if not os.path.exists(REPO_NAME):
        print(f"🚀 Freshly cloning {REPO_NAME} from GitHub...")
        !git clone {REPO_URL}
    %cd {REPO_NAME}
    print(f"Active Directory: {os.getcwd()}")
else:
    print(f"Running in local workspace: {os.getcwd()}")

# Locate and configure dataset.yaml
candidate_yamls = [
    "Food_Detection/DataSet/Version1 Used Data/dataset.yaml",
    os.path.join(os.getcwd(), "Food_Detection", "DataSet", "Version1 Used Data", "dataset.yaml"),
    "/content/SouthIndian_Food_Nutrition/Food_Detection/DataSet/Version1 Used Data/dataset.yaml"
]

dataset_yaml_path = None
for p in candidate_yamls:
    if os.path.exists(p):
        dataset_yaml_path = os.path.abspath(p)
        break

if dataset_yaml_path:
    dataset_dir = os.path.dirname(dataset_yaml_path)
    print(f"✅ Found dataset.yaml at: {dataset_yaml_path}")
    print(f"📁 Dataset root directory: {dataset_dir}")
    
    # Dynamically update `path` in dataset.yaml to match the current running environment
    with open(dataset_yaml_path, "r", encoding="utf-8") as f:
        data_cfg = yaml.safe_load(f)
    
    data_cfg["path"] = dataset_dir
    data_cfg["train"] = "images/train"
    data_cfg["val"] = "images/val"
    data_cfg["test"] = "images/test"
    
    with open(dataset_yaml_path, "w", encoding="utf-8") as f:
        yaml.dump(data_cfg, f, sort_keys=False)
    
    names = data_cfg.get("names", {})
    print(f"\nTotal Classes: {len(names)}")
    if isinstance(names, dict):
        for idx, name in sorted(names.items()):
            print(f"  {idx:2d}: {name}")
    print("\n✅ dataset.yaml configured successfully!")
else:
    raise FileNotFoundError("❌ dataset.yaml not found! Please verify repository.")

## Step 3: Ground-Truth Inspection & Data Visualization
Visualize a sample of training images and verify bounding boxes align accurately with food items.

In [ ]:
import glob, cv2, matplotlib.pyplot as plt

if dataset_yaml_path:
    train_img_dir = os.path.join(dataset_dir, "images", "train")
    train_lbl_dir = os.path.join(dataset_dir, "labels", "train")
    
    img_files = glob.glob(os.path.join(train_img_dir, "*.jpg"))[:4]
    
    if img_files:
        fig, axes = plt.subplots(1, min(4, len(img_files)), figsize=(18, 5))
        if len(img_files) == 1:
            axes = [axes]
            
        for ax, img_path in zip(axes, img_files):
            img = cv2.imread(img_path)
            h, w, _ = img.shape
            stem = os.path.splitext(os.path.basename(img_path))[0]
            lbl_path = os.path.join(train_lbl_dir, f"{stem}.txt")
            
            if os.path.exists(lbl_path):
                with open(lbl_path, 'r') as lf:
                    for line in lf:
                        parts = line.strip().split()
                        if len(parts) >= 5:
                            cid = int(parts[0])
                            xc, yc, bw, bh = map(float, parts[1:5])
                            x1 = int((xc - bw / 2) * w)
                            y1 = int((yc - bh / 2) * h)
                            x2 = int((xc + bw / 2) * w)
                            y2 = int((yc + bh / 2) * h)
                            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
                            cname = names[cid] if isinstance(names, dict) and cid in names else str(cid)
                            cv2.putText(img, cname, (x1, max(y1 - 5, 15)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
            
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            ax.set_title(stem[:20], fontsize=10)
            ax.axis('off')
            
        plt.tight_layout()
        plt.show()
    else:
        print("No sample images found to display.")

## Step 4: Model Selection & Pretrained Initialization
For Food Detection Version 1, we use `yolov8s.pt` (Small) initialized with pretrained COCO weights.

In [ ]:
from ultralytics import YOLO

MODEL_WEIGHTS = 'yolov8s.pt'
print(f"📥 Loading pretrained {MODEL_WEIGHTS}...")
model = YOLO(MODEL_WEIGHTS)
print("✅ Model successfully initialized.")

## Step 5: Train the Food Detection Version 1 Model
Trains YOLOv8 on fast local storage and automatically captures `save_dir` for all outputs.

In [ ]:
import os, shutil

PROJECT = "runs/detect"
EXPERIMENT_NAME = "yolov8s_31classes"
EPOCHS = 100
BATCH_SIZE = 16
IMG_SIZE = 640

print(f"🚀 Starting training on {dataset_yaml_path}...")
print(f"📁 Output directory: {PROJECT}/{EXPERIMENT_NAME}/")

results = model.train(
    data=dataset_yaml_path,
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMG_SIZE,
    patience=20,
    save=True,          # Saves best.pt and last.pt
    save_period=10,     # Milestone checkpoint every 10 epochs
    project=PROJECT,
    name=EXPERIMENT_NAME,
    exist_ok=True,
    device=0 if torch.cuda.is_available() else 'cpu',
    workers=4,
    verbose=True
)

# Deterministic paths from training results
RUN_DIR = str(results.save_dir)
BEST_WEIGHTS_PATH = os.path.join(RUN_DIR, "weights", "best.pt")
LAST_WEIGHTS_PATH = os.path.join(RUN_DIR, "weights", "last.pt")

print("\n🎉 Training Complete!")
print(f"✅ Results saved to  : {RUN_DIR}")
print(f"✅ Best Model       : {BEST_WEIGHTS_PATH}")
print(f"✅ Last Model       : {LAST_WEIGHTS_PATH}")

# Copy to official models directory
DEST_MODELS_DIR = "Food_Detection/Models/Food Detection/Version1"
os.makedirs(DEST_MODELS_DIR, exist_ok=True)

if os.path.exists(BEST_WEIGHTS_PATH):
    shutil.copy2(BEST_WEIGHTS_PATH, os.path.join(DEST_MODELS_DIR, "food_detection_v1_best.pt"))
    shutil.copy2(BEST_WEIGHTS_PATH, os.path.join(DEST_MODELS_DIR, "best.pt"))
    print(f"✅ Copied to: {DEST_MODELS_DIR}/food_detection_v1_best.pt")

if os.path.exists(LAST_WEIGHTS_PATH):
    shutil.copy2(LAST_WEIGHTS_PATH, os.path.join(DEST_MODELS_DIR, "food_detection_v1_last.pt"))
    shutil.copy2(LAST_WEIGHTS_PATH, os.path.join(DEST_MODELS_DIR, "last.pt"))
    print(f"✅ Copied to: {DEST_MODELS_DIR}/food_detection_v1_last.pt")

## Step 6: Review Training Performance & Curves
Visualize the summary `results.png` and `confusion_matrix.png` directly from `RUN_DIR`.

In [ ]:
from IPython.display import Image, display

results_img = os.path.join(RUN_DIR, "results.png")
confusion_img = os.path.join(RUN_DIR, "confusion_matrix.png")

if os.path.exists(results_img):
    print("📈 Training Loss & Validation mAP Curves:")
    display(Image(filename=results_img, width=900))
else:
    print(f"results.png not found at {results_img}")

if os.path.exists(confusion_img):
    print("🎯 Confusion Matrix:")
    display(Image(filename=confusion_img, width=900))

## Step 7: Evaluate Model on Held-Out Test Set
Benchmark generalization performance on the 795 test split images.

In [ ]:
best_eval_path = os.path.join(DEST_MODELS_DIR, "food_detection_v1_best.pt") if os.path.exists(os.path.join(DEST_MODELS_DIR, "food_detection_v1_best.pt")) else BEST_WEIGHTS_PATH

if os.path.exists(best_eval_path):
    print(f"Loading best checkpoint: {best_eval_path}")
    best_model = YOLO(best_eval_path)
    
    print("\nEvaluating on test split...")
    metrics = best_model.val(split='test')
    
    print("\n==========================================")
    print("📊 TEST SET METRICS:")
    print("==========================================")
    print(f"mAP@0.50     : {metrics.box.map50:.4f}")
    print(f"mAP@0.50:0.95: {metrics.box.map:.4f}")
    print(f"Precision    : {metrics.box.mp:.4f}")
    print(f"Recall       : {metrics.box.mr:.4f}")
    print("==========================================")
else:
    print(f"Model weights not found at {best_eval_path}")

## Step 8: Visual Inference on Test Images
Run predictions on unseen test photos and display detection bounding boxes and confidence scores.

In [ ]:
if os.path.exists(best_eval_path) and dataset_yaml_path:
    test_img_dir = os.path.join(dataset_dir, "images", "test")
    sample_tests = glob.glob(os.path.join(test_img_dir, "*.jpg"))[:3]
    
    if sample_tests:
        preds = best_model.predict(source=sample_tests, conf=0.25, iou=0.45)
        
        fig, axes = plt.subplots(1, len(sample_tests), figsize=(18, 6))
        if len(sample_tests) == 1:
            axes = [axes]
            
        for ax, pred in zip(axes, preds):
            annotated_img = pred.plot()
            ax.imshow(cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB))
            ax.set_title("Detection Output", fontsize=12)
            ax.axis('off')
            
        plt.tight_layout()
        plt.show()

## Step 9: Save & Backup Checkpoints and Export ONNX
Exports production ONNX model and prints checkpoint file size verification.

In [ ]:
print("=" * 65)
print("📦 MODEL CHECKPOINTS SUMMARY & ONNX EXPORT")
print("=" * 65)

std_best = os.path.join(DEST_MODELS_DIR, "food_detection_v1_best.pt")
std_last = os.path.join(DEST_MODELS_DIR, "food_detection_v1_last.pt")

if os.path.exists(std_best):
    sz_mb = os.path.getsize(std_best) / (1024 * 1024)
    print(f"✅ BEST MODEL (best.pt) : {sz_mb:.2f} MB")
    print(f"   Path: {std_best}")
    
    # Export to ONNX
    print("\n📦 Exporting best model to ONNX format...")
    try:
        onnx_file = best_model.export(format='onnx', dynamic=True, simplify=True)
        onnx_dest = os.path.join(DEST_MODELS_DIR, "food_detection_v1_best.onnx")
        if os.path.exists(str(onnx_file)) and onnx_file != onnx_dest:
            shutil.copyfile(str(onnx_file), onnx_dest)
            shutil.copyfile(str(onnx_file), os.path.join(DEST_MODELS_DIR, "best.onnx"))
            print(f"✅ ONNX model saved to: {onnx_dest}")
    except Exception as e:
        print(f"Notice on ONNX export: {e}")

if os.path.exists(std_last):
    sz_mb = os.path.getsize(std_last) / (1024 * 1024)
    print(f"✅ LAST MODEL (last.pt) : {sz_mb:.2f} MB")
    print(f"   Path: {std_last}")

print("=" * 65)
print(f"🎉 All checkpoints safely secured in: {DEST_MODELS_DIR}")
print("=" * 65)

## Step 10: 📥 1-Click Download Trained Weights to Your Local Computer
Run this cell in Google Colab to **instantly download `food_detection_v1_best.pt` directly to your local computer's Downloads folder**.

Then move it into `Food_Detection/Models/Food Detection/Version1/` on your PC!

In [ ]:
# 📥 1-Click Download weights with standardized naming: food_detection_v1_best.pt
import os, shutil, glob

STANDARDIZED_BEST = "food_detection_v1_best.pt"
STANDARDIZED_LAST = "food_detection_v1_last.pt"

# Candidate paths for best.pt
candidates = [
    "Food_Detection/Models/Food Detection/Version1/food_detection_v1_best.pt",
    "Food_Detection/Models/Food Detection/Version1/best.pt",
    "runs/detect/yolov8s_31classes/weights/best.pt",
    "runs/detect/Food_Detection/Models/Food Detection/Version1/yolov8s_31classes/weights/best.pt",
    "/content/SouthIndian_Food_Nutrition/runs/detect/Food_Detection/Models/Food Detection/Version1/yolov8s_31classes/weights/best.pt"
] + [p for p in glob.glob("**/best.pt", recursive=True) if "weights" in p]

best_src = next((c for c in candidates if os.path.exists(c) and not os.path.isdir(c)), None)

if best_src:
    shutil.copy2(best_src, STANDARDIZED_BEST)
    print(f"✅ Found best weights at : {best_src}")
    print(f"✅ Standardized model    : {STANDARDIZED_BEST}")
    if "google.colab" in str(get_ipython()):
        from google.colab import files
        print(f"📥 Triggering download of {STANDARDIZED_BEST} to your computer...")
        files.download(STANDARDIZED_BEST)
    else:
        print(f"Saved locally as {STANDARDIZED_BEST}")
else:
    print("⚠️ Training weights not found yet. Please run training step first.")

## Step 11: How to Test with the Streamlit App (`app.py`)
On your local PC:
1. Move the downloaded `food_detection_v1_best.pt` into:
   `Food_Detection/Models/Food Detection/Version1/food_detection_v1_best.pt`
2. Run the Streamlit web app:
```bash
streamlit run app.py
```
3. In the sidebar, select `food_detection_v1_best.pt`.
4. Upload any food image $\rightarrow$ Model 1 detects the dishes $\rightarrow$ Model 2 retrieves ingredients & Home vs Hotel nutrition!